In [1]:
! uv pip install pyyaml

Using Python 3.12.11 environment at: /home/zeus/miniconda3/envs/cloudspace
Checked 1 package in 7ms


In [2]:
# ── Lab 05 - Common Utils and hub.yaml ──
# Result: results/05-config-experiments.md
# Goal  : drive the labs from configs/hub.yaml and common/hf_utils.py
# Needs : HF_READ_TOKEN and HF_WRITE_TOKEN in .env
# Note  : the scratch repo is deleted in the cleanup cell

# ── make `common` importable (labs/ sits next to common/) ──
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from common import hf_utils as hu

# ── config + folders + caches + .env (must run BEFORE the hub imports) ──
cfg = hu.load_config()
paths = hu.setup(cfg)

# ── hub imports ──
import huggingface_hub
from huggingface_hub import HfApi, hf_hub_download

# ── tokens come from the env var NAMES in hub.yaml ──
READ_TOKEN = hu.get_token(cfg, "read")
WRITE_TOKEN = hu.get_token(cfg, "write")

api = HfApi(token=WRITE_TOKEN)
USER = api.whoami()["name"]
CONFIG_REPO = hu.repo_id(cfg, USER, "config_lab")

print("root       :", paths["root"])
print("config repo:", CONFIG_REPO)

root       : /teamspace/studios/this_studio/All-about-finetuning/finetuning/00_platform
config repo: akshayredekar74/config-lab-model


In [3]:
# ── 1. read the config ──
# cfg is a normal Python dict

for section in ["paths", "repos", "release", "switches"]:
    print(f"[{section}]")
    for key, value in cfg[section].items():
        print(f"  {key:16} {value}")
    print()

[paths]
  downloads        datasets/downloads
  upload           datasets/upload
  results          results
  release_source   hub/release
  space_source     hub/spaces/sentiment-analysis

[repos]
  scratch          auth-matrix-scratch
  model            release-lab-model
  space            release-lab-space
  dataset          datasets-lab-tiny
  dataset_files    datasets-lab-files
  config_lab       config-lab-model

[release]
  private          True
  tags             ['v1.0', 'v1.1']
  branch           dev

[switches]
  run_inference    False
  run_gated_test   True
  delete_after     True



In [4]:
# ── 2. experiment 1: switches ──
# Open configs/hub.yaml, set run_inference to true, then re-run
# Cell 1, Cell 2 and this cell. The message changes. No code was edited.

if cfg["switches"]["run_inference"]:
    print("run_inference is ON: this is where the Inference API call would go")
else:
    print("skipped: run_inference is false in hub.yaml (needs paid credits)")

skipped: run_inference is false in hub.yaml (needs paid credits)


In [5]:
# ── 3. experiment 2: override a value in memory ──
# Copy the config, change one name, compare. hub.yaml is untouched.

test_cfg = hu.load_config()
test_cfg["repos"]["model"] = "renamed-model"

print("original:", hu.repo_id(cfg, USER, "model"))
print("changed :", hu.repo_id(test_cfg, USER, "model"))

original: akshayredekar74/release-lab-model
changed : akshayredekar74/renamed-model


In [6]:
# ── 5. experiment 4: release with tags and branch from hub.yaml ──
# Folder = hub/release, tags and branch = release section of hub.yaml

api.create_repo(CONFIG_REPO, private=cfg["release"]["private"], exist_ok=True)

# ── upload the model card folder ──
api.upload_folder(
    repo_id=CONFIG_REPO,
    folder_path=str(paths["release_source"]),
    commit_message="Add release files",
)

# ── one change + one tag per entry in hub.yaml ──
for tag in cfg["release"]["tags"]:
    api.upload_file(
        path_or_fileobj=f"release {tag}\n".encode(),
        path_in_repo="CHANGELOG.md",
        repo_id=CONFIG_REPO,
        commit_message=f"Prepare {tag}",
    )
    api.create_tag(CONFIG_REPO, tag=tag, tag_message=f"Release {tag}", exist_ok=True)

# ── branch from hub.yaml ──
api.create_branch(CONFIG_REPO, branch=cfg["release"]["branch"], exist_ok=True)

refs = api.list_repo_refs(CONFIG_REPO)
tags = [t.name for t in refs.tags]
branches = [b.name for b in refs.branches]
print("tags    :", tags)
print("branches:", branches)

tags    : ['v1.0', 'v1.1']
branches: ['dev', 'main']


In [7]:
# ── 6. each tag holds its own version of CHANGELOG.md ──

seen = {}
for tag in cfg["release"]["tags"]:
    path = hf_hub_download(CONFIG_REPO, "CHANGELOG.md", revision=tag, token=WRITE_TOKEN)
    seen[tag] = Path(path).read_text().strip()
    print(tag, "->", seen[tag])

print("downloads folder:", hu.folder_size_mb(paths["downloads"]), "MB")

CHANGELOG.md:   0%|          | 0.00/13.0 [00:00<?, ?B/s]

v1.0 -> release v1.0


CHANGELOG.md:   0%|          | 0.00/13.0 [00:00<?, ?B/s]

v1.1 -> release v1.1
downloads folder: 6.24 MB


In [8]:
# ── 7. save results ──
# hu.save_result writes into the results folder named in hub.yaml

rows = "\n".join(f"| {tag} | {text} |" for tag, text in seen.items())

text = f"""# 05 - Config Experiments

huggingface_hub {huggingface_hub.__version__}

| Tag | CHANGELOG.md content |
|---|---|
{rows}

Branches: {branches}
Switch run_inference: {cfg["switches"]["run_inference"]}
"""

print("saved:", hu.save_result(cfg, "05-config-experiments.md", text))

saved: /teamspace/studios/this_studio/All-about-finetuning/finetuning/00_platform/results/05-config-experiments.md


In [ ]:
# ── 8. clean up ──
# The delete_after switch in hub.yaml decides

if cfg["switches"]["delete_after"]:
    gone = hu.delete_repo_quietly(api, CONFIG_REPO)
    print("deleted" if gone else "already gone", CONFIG_REPO)
else:
    print("kept", CONFIG_REPO)